# LUT Mapping over AIGs

Map a design to `k`-LUTs using nextmap's AIG backend: build an AIG e-graph, run
equality-saturation optimization (with FRAIG-style simulation to find
equivalences), then cover the AIG with `k`-input LUTs.

Run from the `eval/` directory so `epfl/*.json` resolves.

In [ ]:
import os
import json
import time

import nextmap
import nextmap.db_aig as db_aig
from nextmap.rewrites import aig_opt

TEST_NAME = "adder"
TOP_MODULE = "eval/epfl/adder"
SCHEMA_PATH = os.path.join(os.path.dirname(nextmap.__file__), "schema.sql")

start = time.time()

# Build the AIG e-graph from a Yosys (AIG) JSON netlist.
netlist = db_aig.NetlistDB(SCHEMA_PATH, ":memory:", cnt=10000)
with open(f"epfl/{TEST_NAME}.json") as f:
    netlist.build_from_json(json.load(f)["modules"][TOP_MODULE])

wdsu = nextmap.DisjointSetUnion()
netlist.rebuild(wdsu)

# AIG equality-saturation optimization (rewrites + simulation-based equivalences).
stats = aig_opt.optimize_aig_with_eqsat(netlist, max_iterations=5, verbose=True)
print(f"AIG opt: {stats['total_rewrites']} rewrites over {stats['iterations']} iterations; "
      f"final {stats['final_ands']} ANDs / {stats['final_invs']} INVs")

# k-LUT technology mapping.
nextmap.rewrites.techmap_luts(netlist, k=6, cnt=100, rseed=42, verbose=True)

# Write the LUT-mapped design back out.
os.makedirs("out", exist_ok=True)
out = netlist.write_json()
lut_count = sum(1 for c in out.get("cells", {}).values() if c.get("type") == "$lut")
print(f"Done in {time.time() - start:.1f}s; produced {lut_count} $lut cells")

with open(f"out/lutmapped_{TEST_NAME}.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": out}}, f, indent=2)
